# 4 · Bronze, from documents

A third source, a third shape. This one is a **document store**: MongoDB.

| | |
|---|---|
| **reads** | MongoDB `kerb_app.driver_app_events` |
| **writes** | `teach.bronze_driver_app` in PostgreSQL |
| **runs** | hourly |

A table has a schema the database enforces. **A collection does not.** Two
documents sitting next to each other can have different fields, and nothing
anywhere complains.

That single fact is what this notebook is about.

In [ ]:
import sys; sys.path.insert(0, '.')
from nb import show, sql, fetch, run, counts

import json
import psycopg
from pymongo import MongoClient
from pipelines.lib.config import dsn, SCHEMA, MONGO_URI

collection = MongoClient(MONGO_URI).kerb_app.driver_app_events
print(f'{collection.estimated_document_count():,} documents')
print('event types:', collection.distinct('event_type'))

---

## Step 0 · Look at one document

Not at the schema. There is no schema. Look at **a document**.

In [ ]:
doc = collection.find_one({'event_type': 'trip_offer'})

print(json.dumps(doc, indent=2, default=str))

### Read the shape, not just the values

`app`, `device`, `location` and `payload` are **not values, they are more
documents**. The surge multiplier the pricing team cares about is buried at
`payload.surge_multiplier`.

![](img/docs-1-shape.png)

---

## Step 1 · The flat table we are landing into

Someone has to decide which nested values become columns, and that decision is
this DDL. We take **six fields out of a document that has eleven**.

In [ ]:
DDL = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.bronze_driver_app (
    event_id    TEXT PRIMARY KEY,     -- the app's own id, so a re-read is harmless
    trip_id     TEXT,                 -- how this joins to rides later
    driver_id   TEXT,
    event_type  TEXT,
    happened_at TIMESTAMPTZ,
    app_version TEXT,                 -- nested at app.version
    surge       NUMERIC(6,2)          -- nested at payload.surge_multiplier
);
"""

with psycopg.connect(dsn(), autocommit=True) as c:
    c.execute(DDL)

print('table ready')

---

## Step 2 · Walk a nested path without crashing

`doc['payload']['surge_multiplier']` raises `KeyError` if either level is
missing, and one `KeyError` stops the whole run over a single odd document.

In [ ]:
def dig(doc, path):
    """Follow a nested path, returning None the moment a level is not there."""
    cur = doc
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return None
        cur = cur[key]
    return cur

print('surge       :', dig(doc, ('payload', 'surge_multiplier')))
print('app version :', dig(doc, ('app', 'version')))
print('missing     :', dig(doc, ('pricing', 'surge_multiplier')))
print('nonsense    :', dig(doc, ('payload', 'surge_multiplier', 'deeper')))

Four calls, no exception. `None` means **not found**, and step 4 decides what to
do about that.

---

## Step 3 · The contract, which is a list of places

This is the most important cell in the notebook.

![](img/docs-2-moved.png)

Here is the contract as it was written **in July**, when three paths covered
every document anyone had seen.

In [ ]:
SURGE_PATHS = (
    ('payload', 'surge_multiplier'),      # what the app sends today
    ('pricing', 'surge_multiplier'),      # where we GUESS release 4.2 will move it
    ('surge_multiplier',),                # the old flat shape, still in older documents
)

def surge_of(doc):
    """The surge value, from whichever field this app version happened to use."""
    for path in SURGE_PATHS:
        value = dig(doc, path)
        if value is not None:
            return value
    return None          # none of the known paths matched

today   = {'payload': {'surge_multiplier': 1.4}}
guessed = {'pricing': {'surge_multiplier': 1.4}}
ancient = {'surge_multiplier': 1.4}
unknown = {'payload': {'pricing': {'surgeFactor': 1.4}}}

for name, d in [('today', today), ('guessed', guessed),
                ('ancient', ancient), ('unknown', unknown)]:
    print(f'{name:9} -> {surge_of(d)}')

### Why a list, and not one path

Picture the failure this prevents.

The mobile team ships 4.2 and moves the field. A perfectly reasonable refactor.
They do not tell the data team, because why would they. And then:

| | |
|---|---|
| the pipeline does not fail | it reads the document fine |
| the rows still land | the count is unchanged |
| surge is just empty | on some rows, starting Tuesday |
| nobody notices | for three weeks |

**Every row count check you can write stays green through that.**

---

## Step 4 · Read the documents, and never default a missing value

![](img/docs-3-zero.png)

When `surge_of` returns `None` we **hold** the document. We do **not** write a
zero.

One detail in the query below matters more than it looks.

In [ ]:
def read_batch(paths, limit=20000):
    """Read the NEWEST documents and sort each into landed or held."""
    keep, held = [], []

    # .sort('ts', -1) is not decoration. A collection hands back its OLDEST
    # documents first, and the oldest documents describe rides that fell out of
    # the bronze window weeks ago. Read from the wrong end and every column you
    # worked for arrives full of nulls in silver, with no error anywhere.
    for d in collection.find({'event_type': 'trip_offer'}).sort('ts', -1).limit(limit):
        surge = None
        for path in paths:
            surge = dig(d, path)
            if surge is not None:
                break

        if surge is None:
            held.append(d)
            continue

        keep.append((
            str(d.get('event_id') or d['_id']),
            d.get('trip_id'),
            d.get('driver_id'),
            d.get('event_type'),
            d.get('ts'),
            dig(d, ('app', 'version')),      # also nested, same problem, same solution
            surge))
    return keep, held

keep, held = read_batch(SURGE_PATHS)

print(f'landed : {len(keep):,}')
print(f'held   : {len(held):,}')

## There it is

Documents that match **none** of the three paths we wrote down. Nothing failed.
Nothing logged an error. They are simply held.

**Now look at one.** This is the whole job, in one cell.

In [ ]:
print(json.dumps(held[0], indent=2, default=str))

### Find the value with your eyes

It is there. `payload.pricing.surgeFactor`.

Not the path we guessed. **A different name, at a different depth.** That is the
real shape of this problem: you cannot predict where a field will go. What you
can do is notice, on the day, that some documents match none of your known
paths, because those documents are sitting in front of you with the reason
attached.

## Who is sending it?

In [ ]:
from collections import Counter

versions = Counter(dig(d, ('app', 'version')) for d in held)
for version, n_docs in versions.most_common():
    print(f'  app {version}   {n_docs:>7,} held')

print()
# keep holds tuples, and app_version is the sixth column of each
all_versions = Counter(row[5] for row in keep)
for version, n_docs in sorted(all_versions.items()):
    print(f'  app {version}   {n_docs:>7,} landed')

**One release. Every held document comes from 4.2.0.**

That is the sentence you take to the mobile team, and it took one notebook cell
rather than three weeks.

---

## Step 5 · The fix is one line

Add the path. Nothing else changes.

In [ ]:
SURGE_PATHS = (
    ('payload', 'surge_multiplier'),          # 4.0.6, 4.1.2, 4.1.5
    ('payload', 'pricing', 'surgeFactor'),    # 4.2.0. Found by looking, above
    ('pricing', 'surge_multiplier'),          # what we guessed 4.2 would do. It did not
    ('surge_multiplier',),                    # the old flat shape
)

keep, held = read_batch(SURGE_PATHS)

print(f'landed : {len(keep):,}')
print(f'held   : {len(held):,}')

## And where is the value actually living, across the whole collection?

Ask the collection rather than assuming. This is a real aggregation over all
three hundred thousand documents.

In [ ]:
report = collection.aggregate([
    {'$match': {'event_type': 'trip_offer'}},
    {'$project': {
        'at_payload': {'$cond': [{'$ifNull': ['$payload.surge_multiplier', False]}, 1, 0]},
        'at_nested':  {'$cond': [{'$ifNull': ['$payload.pricing.surgeFactor', False]}, 1, 0]},
        'at_pricing': {'$cond': [{'$ifNull': ['$pricing.surge_multiplier', False]}, 1, 0]},
        'at_flat':    {'$cond': [{'$ifNull': ['$surge_multiplier', False]}, 1, 0]},
    }},
    # a $group output name cannot contain a dot, so the readable names come later
    {'$group': {'_id': None,
                'at_payload': {'$sum': '$at_payload'},
                'at_nested':  {'$sum': '$at_nested'},
                'at_pricing': {'$sum': '$at_pricing'},
                'at_flat':    {'$sum': '$at_flat'},
                'documents':  {'$sum': 1}}},
]).next()

NAMES = {'at_payload': 'payload.surge_multiplier',
         'at_nested':  'payload.pricing.surgeFactor',
         'at_pricing': 'pricing.surge_multiplier',
         'at_flat':    'surge_multiplier'}

total = report['documents']
print(f'in {total:,} documents, the surge value lives at:\n')
for key, label in NAMES.items():
    n_found = report[key]
    print(f'  {label:30} {n_found:>8,}   {n_found / total:>6.1%}')

**Four documents at the path we guessed.** Somebody on the mobile team started
the refactor we expected, then changed their mind. That is a report worth having
on a wall.

---

## Step 6 · Write in one batch, not one row at a time

In [ ]:
INSERT = f"""
    INSERT INTO {SCHEMA}.bronze_driver_app
        (event_id, trip_id, driver_id, event_type, happened_at, app_version, surge)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (event_id) DO NOTHING
"""

import time
t0 = time.time()

with psycopg.connect(dsn(), autocommit=False) as c, c.cursor() as cur:
    cur.executemany(INSERT, keep)        # ONE connection, ONE transaction
    c.commit()

print(f'wrote {len(keep):,} rows in {time.time() - t0:.2f}s')

### A story worth telling in class

The first version of this pipeline opened **a fresh database connection for
every held record**. Forty thousand of them took **212 seconds**. Batching made
it **0.7**. Same code, same result, three hundred times faster.

The lesson is not *"batch your writes"*. It is:

> **When something is inexplicably slow, count how many times you are opening a
> connection.**

---

## And now the packaged pipeline, over the whole collection

In [ ]:
run('-m', 'pipelines.p3_bronze_driver_app')

In [ ]:
sql(f"""
    SELECT pipeline, status, rows_in, rows_out,
           round(extract(epoch from (ended_at - started_at))::numeric, 2) AS secs, message
    FROM {SCHEMA}.runs
    WHERE pipeline = 'p3_bronze_driver_app'
    ORDER BY started_at DESC LIMIT 3
""", 'the run log')

In [ ]:
sql(f"""
    SELECT app_version, count(*) AS events, round(avg(surge), 3) AS avg_surge
    FROM {SCHEMA}.bronze_driver_app
    GROUP BY 1 ORDER BY 2 DESC
""", 'what landed, by app version')

**That average is only trustworthy because nothing was defaulted.** Every row
behind it had a surge value we actually found.

In notebook 9 you will move the field, run this again, and watch the held pile
fill up while the row count stays exactly the same.

---

## What you learned

- A collection **has no schema**. Two documents can disagree, and nothing complains
- Bronze takes **the fields the business agreed on**, not every field that exists
- Walk nested paths with something that returns `None`, never with `[]`
- A contract for documents is **a list of every path a value has legitimately
  lived at**
- **Zero is not the same as unknown.** Holding costs a day. Defaulting costs
  a wrong number forever
- Ask the source **which shape it is actually sending**, rather than assuming
- When something is inexplicably slow, count your connections